# SI100B Lecture 4–6 Review Workbook

配合习题课 Recitation 3 使用。每个小节先说明这一段代码在做什么，再给出可以运行的代码，结果写在注释里。

从上往下运行（`Shift + Enter`）即可。前面的小节是回顾与演示，**Exercises** 是练习（只有题目），
**Check Your Answers** 逐题给出应有结果，**Reference Solutions** 是参考代码。

为了能反复运行，演示代码里的输入都写成变量；写作业时换成 `input()`（写法见 Recitation 2+ 的 Notebook）。

## Approximation Methods

### 用容差（epsilon）定义"足够接近"

猜不出来精确答案时，就换一个标准：误差小于某个我们自己定的上限就算成功。
`abs(guess**2 - x) >= epsilon` 表示"还不够近"，循环继续；`guess` 每轮增加一个固定的步长 `increment`。
注意循环体里必须先更新 `guess` 再计数，否则计数会多算一次。

In [ ]:
x = 36
epsilon = 0.01
increment = 0.0001

guess = 0.0
num_guesses = 0
while abs(guess**2 - x) >= epsilon:
    guess += increment
    num_guesses += 1

print('num_guesses =', num_guesses)
print(guess, 'is close to square root of', x)

### 步长决定精度上限

步长至少要小于容差，否则每次都可能跨过答案。这里把步长从 `0.0001` 调成 `0.00001`，
结果更精确，但猜测次数变成十倍：算 `6 / increment` 就能估算次数。

In [ ]:
x = 36
epsilon = 0.01
increment = 0.00001

guess = 0.0
num_guesses = 0
while abs(guess**2 - x) >= epsilon:
    guess += increment
    num_guesses += 1

print('num_guesses =', num_guesses)
print(round(guess, 9), 'is close to square root of', x)

### 反例：猜测越过答案（overshoot）

`sqrt(54321)` 不是整数。`guess` 每次加 `0.0001`，会在 233 附近"跨过"答案：跨过之前平方偏小，
跨过之后平方偏大，误差始终是几十，永远不会小于 `0.01`，所以原来的循环不会退出。

在 Notebook 里演示这种循环必须自己加上限，否则单元格会一直跑下去。下面用 `numGuesses < 200000`
当保险，观察误差从来没有落进容差以内。

In [ ]:
x = 54321
epsilon = 0.01
increment = 0.0001

guess = 0.0
numGuesses = 0
while abs(guess**2 - x) >= epsilon and numGuesses < 200000:
    guess += increment
    numGuesses += 1

print('跑了', numGuesses, '次仍然没满足容差')
print('此时 guess =', round(guess, 4), ' guess**2 - x =', round(guess**2 - x, 4))

### 修法一：给循环加上界

在条件里补上 `and guess**2 <= x`：平方一旦超过 `x` 就停止。循环能退出了，
但程序会明确报告失败——这比死循环好，不过仍然没算出答案。

In [ ]:
x = 54321
epsilon = 0.01
increment = 0.0001

guess = 0.0
numGuesses = 0
while abs(guess**2 - x) >= epsilon and guess**2 <= x:
    guess += increment
    numGuesses += 1

print('numGuesses =', numGuesses)
if abs(guess**2 - x) >= epsilon:
    print('Failed on square root of', x)
else:
    print(guess, 'is close to square root of', x)

### 二分查找：每次丢掉一半

只要满足两个前提——搜索空间有序、能判断猜测偏大还是偏小——就可以每轮把候选区间减半。
下面维护 `[low, high]`，取中点，按 `guess**2` 与 `x` 的大小关系舍弃一半。
`epsilon = 0.01` 时只需要几十次，而逐次加步长要两千万次以上。

In [ ]:
x = 54321
epsilon = 0.01

low = 0
high = x
guess = (high + low)/2.0
num_guesses = 0
while abs(guess**2 - x) >= epsilon:
    if guess**2 < x:
        low = guess
    else:
        high = guess
    guess = (high + low)/2.0
    num_guesses += 1

print('num_guesses =', num_guesses)
print(guess, 'is close to square root of', x)

### 追踪二分查找的每一轮

把 `low`、`high`、`guess` 每轮都打印出来，可以看到区间长度每轮减半。
用 `x = 25` 演示：第一轮猜 12.5（偏大），于是把上界压到 12.5；下一轮猜 6.25，依此类推。

In [ ]:
x = 25
epsilon = 0.01

low = 0
high = x
guess = (high + low)/2.0
print('%-6s %-9s %-9s %-9s' % ('轮次', 'low', 'high', 'guess'))
print('%-6s %-9.4f %-9.4f %-9.4f' % ('初始', low, high, guess))

num_guesses = 0
while abs(guess**2 - x) >= epsilon:
    if guess**2 < x:
        low = guess
    else:
        high = guess
    guess = (high + low)/2.0
    num_guesses += 1
    print('%-6d %-9.4f %-9.4f %-9.4f' % (num_guesses, low, high, guess))

print('共', num_guesses, '轮，guess =', guess)

### 边界情况：0 < x < 1

`0 < x < 1` 时 `sqrt(x) > x`，答案不在 `[0, x]` 里。要按 `x` 是否大于 1 分别设置初始区间：
`x >= 1` 用 `[1, x]`，否则用 `[x, 1]`。下面求 `0.5` 的平方根，答案约 `0.707`。

In [ ]:
x = 0.5
epsilon = 0.01

if x >= 1:
    low = 1.0
    high = x
else:
    low = x
    high = 1.0
guess = (high + low)/2

while abs(guess**2 - x) >= epsilon:
    if guess**2 < x:
        low = guess
    else:
        high = guess
    guess = (high + low)/2.0

print(f'{str(guess)} is close to square root of {str(x)}')

### 牛顿—拉夫森法：用导数迭代

求 `k` 的平方根等价于求 `p(x) = x**2 - k` 的根，迭代式 `guess = guess - (guess**2 - k) / (2*guess)`
里的 `2*guess` 就是 `p(x)` 的导数。从一个不太准的初值 `k/2` 出发，几次就收敛。

In [ ]:
epsilon = 0.01
k = 24.0
guess = k/2.0
num_guesses = 0

while abs(guess*guess - k) >= epsilon:
    num_guesses += 1
    guess = guess - (((guess**2) - k)/(2*guess))
print('num_guesses = ' + str(num_guesses))
print('Square root of ' + str(k) + ' is about ' + str(guess))

### 同一个问题、三种算法的次数对比

固定 `x = 36`、`epsilon = 0.01`，分别用增量逼近、二分查找、牛顿法求平方根，数一数各要多少次。
这张对照比"记住哪个快"更有用：它说明提速来自算法结构，而不是更努力的循环。

In [ ]:
x = 36
epsilon = 0.01

# 增量逼近
guess = 0.0
inc_guesses = 0
while abs(guess**2 - x) >= epsilon:
    guess += 0.0001
    inc_guesses += 1

# 二分查找
low, high = 1.0, x
guess_b = (high + low)/2.0
bis_guesses = 0
while abs(guess_b**2 - x) >= epsilon:
    if guess_b**2 < x:
        low = guess_b
    else:
        high = guess_b
    guess_b = (high + low)/2.0
    bis_guesses += 1

# 牛顿法
guess_n = x/2.0
newton_guesses = 0
while abs(guess_n**2 - x) >= epsilon:
    newton_guesses += 1
    guess_n = guess_n - (((guess_n**2) - x)/(2*guess_n))

print('增量逼近', inc_guesses, '次')
print('二分查找', bis_guesses, '次')
print('牛顿法  ', newton_guesses, '次')

## Functions

### 定义与规格：先写清输入输出

`def` 定义函数，函数名后面是形参列表；紧跟的三引号字符串是规格（specs），写清输入与返回。
函数体里的 `return` 决定调用表达式的值。定义本身不会执行函数体，只有调用才会。

In [ ]:
def is_even(i):
    """
    Input: i, a positive int
    Returns True if i is even, otherwise False
    """
    return i%2 == 0

print(is_even(3))
print(is_even(8))

### print 与 return 的区别

`print` 把内容写到屏幕，`return` 把值交给调用者。一个函数可以打印很多行却什么都不返回，
也可以返回一个值却什么都不打印。下面用 `add` 与 `mult` 对照：`add(1, 2)` 的返回值被直接丢掉，
所以什么都不会打印；`mult(3, 4)` 内部打印了 12，但它自己返回的是 `None`。

In [ ]:
def add(x, y):
    return x + y

def mult(x, y):
    print(x*y)

add(1, 2)
print(add(2, 3))
mult(3, 4)
print(mult(4, 5))

### 没有 return 的函数返回 None

函数体里没有 `return` 时，Python 自动返回 `None`（`return None` 与裸 `return` 效果相同）。
`None` 表示"没有值"，交互式环境里不会打印它；一旦拿它参与运算就会在别处报错。
下面把返回值接住再看类型，就能看清这一点。

In [ ]:
def does_nothing(i):
    """
    Input: i, a positive int
    Does not return anything
    """
    i%2 == 0

result = does_nothing(3)
print(result)
print(result is None)

### 把二分求根封装成函数

把上一节的二分代码包进函数：`epsilon` 留在函数体里，调用者只给 `x`。
封装之后同一个函数换不同实参就能复用，调用者也不需要知道内部用的是二分法。

In [ ]:
def bisection_root(x):
    epsilon = 0.01
    low = 0
    high = x
    ans = (high + low)/2.0
    while abs(ans**2 - x) >= epsilon:
        if ans**2 < x:
            low = ans
        else:
            high = ans
        ans = (high + low)/2.0
    return ans

print(bisection_root(4))
print(bisection_root(123))

### 模块化：底层函数 + 上层函数

再写一个调用它的函数。下面这个函数数出 0 到 `(n+1)**2` 之间有多少个整数的平方根落在
`n` 的 `epsilon` 邻域内，它把"算平方根"交给 `bisection_root`，自己只负责计数。
`n = 10`、`epsilon = 0.1` 时答案是 4（99、100、101、102）。

In [ ]:
def count_nums_with_sqrt_close_to(n, epsilon):
    """ n is an int > 2
      epsilon is a positive number < 1
      Returns how many integers have a square
      root within epsilon of n """
    count = 0
    for i in range((n + 1)**2):
        if abs(bisection_root(i) - n) < epsilon:
            count += 1
    return count

print(count_nums_with_sqrt_close_to(10, 0.1))

### 默认参数

有些参数几乎每次都用同一个值，可以在定义里给默认值。**默认参数必须写在参数列表最后**，
否则调用时位置实参和关键字实参的顺序会出错。下面 `epsilon` 默认 0.01，也可以显式传更小的值。

In [ ]:
def bisection_root_new(x, epsilon = 0.01):
    low = 0
    high = x
    ans = (high + low)/2.0
    while abs(ans**2 - x) >= epsilon:
        if ans**2 < x:
            low = ans
        else:
            high = ans
        ans = (high + low)/2.0
    return ans

print(bisection_root_new(123))                    # 用默认 epsilon
print(bisection_root_new(123, 0.001))             # 位置传参
print(bisection_root_new(123, epsilon = 0.001))   # 关键字传参
print(bisection_root_new(0.001, 123))             # 不报错，但含义完全错了

### 作用域：函数内外的同名变量互不影响

函数调用会创建独立的环境。下面函数内外各有一个 `x`：函数里的 `x = x + 1` 只改局部那个，
返回后局部环境消失，全局 `x` 仍然是 3，而 `z` 拿到了返回值 4。

In [ ]:
def f(x):
    x = x + 1
    print('in f(x): x =', x)
    return x

x = 3
z = f(x)
print('global x =', x, ' z =', z)

### 读外部变量可以，直接改不行

函数里可以读全局变量；但只要函数体里对某个名字有赋值，整个函数内它都算局部变量。
下面 `f` 新建了局部 `x`，`g` 只读全局 `x`，`h` 里的 `x += 1` 会把 `x` 变成局部的，
读它时还没赋值，于是报 `UnboundLocalError`。

In [ ]:
def f(y):
    x = 1
    x += 1
    print(x)

def g(y):
    print(x)
    print(x + 1)

def h(y):
    x += 1

x = 5
f(x)          # 内部新建局部 x：打印 2，全局 x 不变
print(x)      # 5

g(x)          # 只读全局 x：打印 5、6
print(x)      # 仍然是 5

# h(x) 会报 UnboundLocalError：x += 1 让 x 在整个函数内都是局部变量，读它时还没赋值
# print(h(x))

### LEGB：名字查找顺序

Python 查一个名字的顺序是 Local → Enclosing-function → Global → Built-in：先在函数局部找，
再找外层函数，再找模块顶层，最后找内置名字。下面 `abs` 与 `len` 都是内置层次的名字，
函数里可以直接用。

In [ ]:
def describe(i):
    n = len(str(abs(i)))
    return (i, n)

print(describe(-12345))
print(describe(7))

### 案例：range 含头不含尾导致的差一错

同一个 `sum_odd` 的 `for` 版与 `while` 版第一次测试就不一致：`for` 版用 `range(a, b)` 只加到
`b - 1`，得到 5；`while` 版用 `i <= b` 把 `b` 也加上，得到 9。把 `range` 改成 `range(a, b + 1)`
两版才一致。下面把两个版本并排跑出来对照。

In [ ]:
def sum_odd_for(a, b):
    sum_of_odds = 0
    for i in range(a, b):
        sum_of_odds += i
    return sum_of_odds

def sum_odd_while(a, b):
    sum_of_odds = 0
    i = a
    while i <= b:
        sum_of_odds += i
        i += 1
    return sum_of_odds

print('for  :', sum_odd_for(2, 4))
print('while:', sum_odd_while(2, 4))

def sum_odd_fixed(a, b):
    sum_of_odds = 0
    for i in range(a, b + 1):
        if i%2 == 1:
            sum_of_odds += i
    return sum_of_odds

print('fixed(2, 4):', sum_odd_fixed(2, 4))
print('fixed(2, 7):', sum_odd_fixed(2, 7))

## Functions as Arguments

### 函数也是对象，可以绑定到新名字

函数有类型（`function`），可以出现在赋值语句右边：`my_func = is_even` 只是把名字绑到同一个函数对象上，
不是调用。之后 `my_func(4)` 与 `is_even(4)` 完全等价。

In [ ]:
def is_even(i):
    return i%2 == 0

my_func = is_even      # 别名，不调用
a = is_even(3)
b = my_func(4)
print(a, b)
print(type(is_even))

### 把函数作为参数传进去

传函数时只写函数名、不加括号；真正调用发生在被调用函数内部（`op(x, y)`）。
下面 `calc` 收到 `add` 之后才执行它，所以 `calc(add, 2, 3)` 得到 5。

In [ ]:
def calc(op, x, y):
    return op(x, y)

def add(a, b):
    return a + b

def div(a, b):
    if b != 0:
        return a/b
    print("Denom was 0.")

print(calc(add, 2, 3))
res = calc(div, 2, 0)
print(res)

### 函数作为返回值

内层定义的函数可以像其他对象一样被返回。`make_prod(2)` 返回内层函数 `g`，
它执行时仍然看得到外层作用域里的 `a = 2`。两种写法等价：连写两次调用，或先把函数存进变量。

In [ ]:
def make_prod(a):
    def g(b):
        return a*b
    return g

print(make_prod(2)(3))

doubler = make_prod(2)
print(doubler(3))

### lambda：不给名字的函数

只用一次、逻辑很短的函数可以写成 `lambda 参数: 表达式`，它等价于一个具名函数，
但**没有 `return` 关键字**，也不绑定名字，所以无法复用。下面用 `apply` 把"数一遍"和
"判断标准"拆开：判断标准作为参数传进来，换成 lambda 结果不变。

In [ ]:
def apply(criteria, n):
    """ criteria is a func that takes in a number and returns a bool
    n is an int
    Returns how many ints from 0 to n (inclusive) match the criteria """
    count = 0
    for i in range(n + 1):
        if criteria(i):
            count += 1
    return count

def is_even(x):
    return x % 2 == 0

print(apply(is_even, 10))
print(apply(lambda x: x%2 == 0, 10))

### 高阶过程的另一个例子：do_twice

`do_twice(n, fn)` 把 `fn` 连续作用两次：先算 `fn(n)`，再对结果调用一次 `fn`。
传进 `lambda x: x**2` 时，`3` 先变成 `9`，再变成 `81`。

In [ ]:
def do_twice(n, fn):
    return fn(fn(n))

print(do_twice(3, lambda x: x**2))
print(do_twice(5, lambda x: x + 1))

### 元组：不可变的有序序列

元组用逗号分隔元素，**决定元组的是逗号而不是括号**：`(2,)` 是一个元素的元组，`(2)` 只是整数。
元组不可变，所以不能给某个位置重新赋值；索引从 0 开始，切片同样含头不含尾。

In [ ]:
hello_tuple = (2, "stu", 3)
print(hello_tuple[0])
print(hello_tuple[1:3])
print((2, "stu", 3) + (5, 6))
print((1, 2) * 2)
print(len(hello_tuple), max((3, 5, 0)))
# hello_tuple[1] = 4      # 取消注释会报 TypeError

### 元组的索引与切片

`seq[3]` 取出第 4 个元素（本身又是元组），`seq[3][0]` 可以接着往下取。
切片越界不报错，索引越界报 `IndexError`；元组也能用 `for` 逐个遍历。

In [ ]:
seq = (2, "a", 4, (1, 2))
print(len(seq))
print(seq[3], seq[-1], seq[3][0])
print(seq[-2:])
print(seq[1:4:2])
print(seq[:-1])
for e in seq:
    print(e, end=' ')
print()

### 解包与交换

等号右边是元组时，左边的多个名字会依次接住它的元素，这叫解包。交换两个变量不需要临时变量：
`x, y = y, x` 会先把右边打包成元组再解包。括号可以省略。

In [ ]:
x, y = 1, 2
x, y = y, x
print(x, y)

(a, b) = (7, 8)
print(a, b)

### 用元组返回多个值

函数一次只能返回一个对象，那个对象可以是元组，于是看起来像返回了多个值。
调用处可以直接解包：`(quot, rem) = quotient_and_remainder(5, 2)`。左侧名字个数必须与元素个数一致。

In [ ]:
def quotient_and_remainder(x, y):
    q = x // y
    r = x % y
    return (q, r)

both = quotient_and_remainder(10, 3)
print(both)

(quot, rem) = quotient_and_remainder(5, 2)
print(quot, rem)

### 可变数量的参数

形参名前加 `*` 表示收集所有位置实参，函数体里这个参数就是一个元组，所以能遍历、能取 `len`。
`mean(1, 2, 3, 4, 5, 6)` 的和是 21，除以 6 得到 3.5。

In [ ]:
def mean(*args):
    tot = 0
    for a in args:
        tot += a
    return tot / len(args)

print(mean(1, 2, 3, 4, 5, 6))
print(mean(10, 20))

## Exercises

下面 15 道题覆盖前面各节。每题只给要求和占位变量，自己写完后运行 **Check Your Answers** 逐题对照，
想不出来再看 **Reference Solutions**。

| 题号 | 考察内容 |
|:---:|:---|
| 1 – 2 | 近似求解与二分查找 |
| 3 – 4 | 函数定义、`print` 与 `return` |
| 5 – 6 | 参数、返回值 |
| 7 – 8 | 默认参数与作用域 |
| 9 – 10 | 函数复用与元组 |
| 11 – 12 | 多返回值与高阶过程 |
| 13 – 15 | 整除判断、立方根与字符串元组 |

In [ ]:
# 练习 1：用增量逼近求 x = 49 的平方根（epsilon = 0.01，increment = 0.0001），一共猜了多少次？（填整数）
x = 49
epsilon = 0.01
increment = 0.0001
ans_1 = None

In [ ]:
# 练习 2：用二分查找求 x = 49 的平方根（epsilon = 0.01），循环执行了多少轮？（填整数）
x = 49
epsilon = 0.01
ans_2 = None

In [ ]:
# 练习 3：写一个函数 is_even(i) 判断偶数，然后填上 is_even(4) 的结果（填 True 或 False）
ans_3 = None

In [ ]:
# 练习 4：下面三行会打印什么？（把完整输出拼成一个字符串，行与行之间用 \n）
def mult(x, y):
    print(x*y)

print(mult(3, 4))
ans_4 = None

In [ ]:
# 练习 5：写一个函数 square(x) 返回 x 的平方，然后填上 square(7) 的结果（填整数）
ans_5 = None

In [ ]:
# 练习 6：写一个函数 avg3(a, b, c) 返回三个数的平均值，然后填上 avg3(1, 2, 3) 的结果（填数字）
ans_6 = None

In [ ]:
# 练习 7：写函数 power(x, n = 2) 返回 x 的 n 次方，然后填上 power(5) 与 power(2, 5) 的结果（用空格连接）
ans_7 = None

In [ ]:
# 练习 8：运行下面代码，函数 f 内部打印的那一行是什么？（原样填字符串，不要引号）
def f(x):
    x = x + 1
    print('in f(x): x =', x)
    return x

x = 3
z = f(x)
ans_8 = None

In [ ]:
# 练习 9：用 bisection_root(x) 求 2 的平方根，把结果保留 2 位小数（填字符串，例如 "1.23"）
def bisection_root(x):
    epsilon = 0.01
    low = 0
    high = x
    ans = (high + low)/2.0
    while abs(ans**2 - x) >= epsilon:
        if ans**2 < x:
            low = ans
        else:
            high = ans
        ans = (high + low)/2.0
    return ans

ans_9 = None

In [ ]:
# 练习 10：seq = (2, "a", 4, (1, 2))，seq[-2:] 的结果是什么？（原样填元组）
seq = (2, "a", 4, (1, 2))
ans_10 = None

In [ ]:
# 练习 11：写函数 quotient_and_remainder(x, y) 返回 (商, 余数)，然后填上 (17, 5) 的结果（填元组）
ans_11 = None

In [ ]:
# 练习 12：用 apply(criteria, n) 统计 0 到 10 之间偶数的个数（填整数）
def apply(criteria, n):
    count = 0
    for i in range(n + 1):
        if criteria(i):
            count += 1
    return count

ans_12 = None

In [ ]:
# 练习 13：写函数 div_by(n, d)，判断 d 能否整除 n。然后填上 div_by(10, 3) 与 div_by(195, 13) 的结果（用空格连接）
def div_by(n, d):
    """ n and d are ints > 0
    Returns True if d divides n evenly and False otherwise """
    # 在这里写你的代码

ans_13 = None

In [ ]:
# 练习 14：用下面的函数求 27 的立方根，把结果保留 2 位小数（填字符串，例如 "1.23"）
def bisection_cuberoot(x):
    epsilon = 0.01
    low = 0
    high = x
    ans = (high + low)/2.0
    while abs(ans**3 - x) >= epsilon:
        if ans**3 < x:
            low = ans
        else:
            high = ans
        ans = (high + low)/2.0
    return ans

ans_14 = None

In [ ]:
# 练习 15：写函数 char_counts(s)，返回 (元音个数, 辅音个数)。然后填上 char_counts("hello") 的结果（填元组）
def char_counts(s):
    """ s is a string of lowercase chars
    Return a tuple where the first element is the number of vowels
    in s and the second element is the number of consonants in s """
    # 在这里写你的代码

ans_15 = None

## Check Your Answers

运行下面的单元格，它会逐题显示**你的结果**和**应有结果**。没做的题显示「未完成」。
比较前会自动去掉首尾空格。

In [ ]:
expected = {
    1: "69993",
    2: "14",
    3: "True",
    4: "12\nNone",
    5: "49",
    6: "2.0",
    7: "25 32",
    8: "in f(x): x = 4",
    9: "1.41",
    10: "(4, (1, 2))",
    11: "(3, 2)",
    12: "6",
    13: "False True",
    14: "3.00",
    15: "(2, 3)",
}

mine = {
    1: ans_1,
    2: ans_2,
    3: ans_3,
    4: ans_4,
    5: ans_5,
    6: ans_6,
    7: ans_7,
    8: ans_8,
    9: ans_9,
    10: ans_10,
    11: ans_11,
    12: ans_12,
    13: ans_13,
    14: ans_14,
    15: ans_15,
}

right = 0
for k in range(1, 16):
    got = str(mine[k]).strip()
    want = str(expected[k]).strip()
    if got == want:
        right += 1
        print(k, "正确   ", want)
    else:
        shown = "未完成" if got in ("None", "") else got
        print(k, "你的结果：", shown, "   应有结果：", want)
print()
print("做对", right, "/", len(expected), "题")

做错或没做的题，回看对应小节：

- 第 1 – 2 题：Approximation Methods（增量逼近、二分查找）
- 第 3 – 4 题：Functions（定义、`print` 与 `return`）
- 第 5 – 6 题：Functions（参数与返回值）
- 第 7 – 8 题：Functions（默认参数、作用域）
- 第 9 – 10 题：Functions（函数复用）、Functions as Arguments（元组）
- 第 11 – 12 题：Functions as Arguments（多返回值、高阶过程）
- 第 13 – 15 题：Functions as Arguments（整除判断、立方根、字符串元组）

## Reference Solutions

每题先说明思路，再给可以运行的代码。建议先自己写过再看。

### 第 1 题：数增量逼近的次数

循环条件是"误差还没落进容差"。`49` 是完全平方数，`guess` 加到 `6.9993` 时误差已经小于 `0.01`，
所以一共 **69993 次**（约 7 万）。

In [ ]:
x = 49
epsilon = 0.01
increment = 0.0001

guess = 0.0
num_guesses = 0
while abs(guess**2 - x) >= epsilon:
    guess += increment
    num_guesses += 1
print(num_guesses)

### 第 2 题：数二分查找的轮数

二分查找每轮把区间减半，从 `[0, 49]` 出发大约 `log2(49/0.005)` 轮就能把误差压到 `0.01` 以下。

In [ ]:
x = 49
epsilon = 0.01

low = 0
high = x
guess = (high + low)/2.0
rounds = 0
while abs(guess**2 - x) >= epsilon:
    if guess**2 < x:
        low = guess
    else:
        high = guess
    guess = (high + low)/2.0
    rounds += 1
print(rounds)

### 第 3 题：判断偶数

`i % 2 == 0` 就是"除以 2 余 0"，返回布尔值。`4` 是偶数，所以结果是 `True`。

In [ ]:
def is_even(i):
    return i%2 == 0

print(is_even(4))

### 第 4 题：print 的输出与返回值

`mult(3, 4)` 先在函数内部打印 `12`，然后返回 `None`；外层的 `print` 再把这个 `None` 打印出来。
所以屏幕上是两行：`12` 和 `None`。

In [ ]:
def mult(x, y):
    print(x*y)

print(mult(3, 4))

### 第 5 题：平方函数

一行就能写完：`return x * x`。`square(7)` 得到 `49`。

In [ ]:
def square(x):
    return x * x

print(square(7))

### 第 6 题：三个数的平均值

先求和再除以 3。`1 + 2 + 3 = 6`，`6 / 3 = 2.0`：`/` 的结果一定是 `float`。

In [ ]:
def avg3(a, b, c):
    return (a + b + c) / 3

print(avg3(1, 2, 3))

### 第 7 题：带默认参数的幂函数

`n` 的默认值是 2，不传就按平方算。`power(5)` 得到 `25`，`power(2, 5)` 得到 `32`。

In [ ]:
def power(x, n = 2):
    result = 1
    for i in range(n):
        result *= x
    return result

print(power(5), power(2, 5))

### 第 8 题：函数内部的打印

函数内 `x = x + 1` 改的是局部 `x`（由 3 变成 4），打印出的就是这一行。
全局 `x` 仍然是 3，`z` 拿到返回值 4。

In [ ]:
def f(x):
    x = x + 1
    print('in f(x): x =', x)
    return x

x = 3
z = f(x)
print('global x =', x, 'z =', z)

### 第 9 题：复用二分求根函数

直接调用 `bisection_root(2)`，再用 f-string 的 `:.2f` 保留两位小数，得到 `1.41`。

In [ ]:
def bisection_root(x):
    epsilon = 0.01
    low = 0
    high = x
    ans = (high + low)/2.0
    while abs(ans**2 - x) >= epsilon:
        if ans**2 < x:
            low = ans
        else:
            high = ans
        ans = (high + low)/2.0
    return ans

print(f'{bisection_root(2):.2f}')

### 第 10 题：元组切片

`seq[-2:]` 表示"从倒数第二个元素取到结尾"，也就是最后两个元素：`4` 和 `(1, 2)`。

In [ ]:
seq = (2, "a", 4, (1, 2))
print(seq[-2:])

### 第 11 题：用元组返回商和余数

`//` 得商、`%` 得余数，打包成元组返回。`17 // 5 = 3`，`17 % 5 = 2`。

In [ ]:
def quotient_and_remainder(x, y):
    q = x // y
    r = x % y
    return (q, r)

print(quotient_and_remainder(17, 5))

### 第 12 题：用高阶过程统计偶数

把判定标准写成 `lambda x: x % 2 == 0` 传进 `apply`，数出 0 到 10 之间的偶数个数：0、2、4、6、8、10 共 6 个。

In [ ]:
def apply(criteria, n):
    count = 0
    for i in range(n + 1):
        if criteria(i):
            count += 1
    return count

print(apply(lambda x: x % 2 == 0, 10))

### 第 13 题：判断能否整除

`n % d == 0` 就是"除以 d 余 0"，直接返回这个布尔值。`10 % 3` 是 1（False），`195 % 13` 是 0（True）。

In [ ]:
def div_by(n, d):
    """ n and d are ints > 0
    Returns True if d divides n evenly and False otherwise """
    return n % d == 0

print(div_by(10, 3), div_by(195, 13))

### 第 14 题：二分查找求立方根

和求平方根唯一的区别是把 `ans**2` 换成 `ans**3`。`x >= 1` 时立方根落在 `[0, x]` 里，初始区间不用改。用 `:.2f` 格式化得到 `3.00`。

In [ ]:
def bisection_cuberoot(x):
    epsilon = 0.01
    low = 0
    high = x
    ans = (high + low)/2.0
    while abs(ans**3 - x) >= epsilon:
        if ans**3 < x:
            low = ans
        else:
            high = ans
        ans = (high + low)/2.0
    return ans

print(f'{bisection_cuberoot(27):.2f}')

### 第 15 题：数元音与辅音

遍历字符串，用 `in` 判断字符是否属于 `'aeiou'`，两个计数器分别累加，最后打包成元组返回。`"hello"` 里元音是 `e`、`o`（2 个），辅音是 `h`、`l`、`l`（3 个）。

In [ ]:
def char_counts(s):
    """ s is a string of lowercase chars
    Return a tuple where the first element is the number of vowels
    in s and the second element is the number of consonants in s """
    vowels = 0
    consonants = 0
    for ch in s:
        if ch in 'aeiou':
            vowels += 1
        else:
            consonants += 1
    return (vowels, consonants)

print(char_counts("hello"))

## Common Pitfalls

写作业前扫一眼。表里的现象在前面各节都出现过，这里只做汇总，不重复讲解。

| 易错点 | 正确做法 |
|:---|:---|
| 用 `==` 判断循环该不该退出 | 用 `abs(guess**2 - x) >= epsilon` 这类区间判据 |
| 循环条件可能永远不满足 | 加上界条件（`and guess**2 <= x`）或换算法，避免死循环 |
| 步长比容差还大 | 先估算 `x / increment` 的量级，再决定参数 |
| 以为函数定义会执行函数体 | 只有调用（`f(...)`）才会执行 |
| 把函数名和函数调用混起来 | 传函数作参数写 `f`，要取结果写 `f(...)` |
| 函数没有 `return` 却去用返回值 | 没有 `return` 时返回 `None` |
| `print` 当成 `return` | `print` 只写屏幕，`return` 才把值交给调用者 |
| 默认参数写在参数列表中间 | 默认参数必须放在最后 |
| 在函数里直接改全局变量 | 通过参数传入、用返回值传出；`global` 不推荐 |
| `range(a, b)` 少算一个 | 含头不含尾，要含 `b` 就写 `b + 1` |
| 以为 `(2)` 是元组 | 决定元组的是逗号，写 `(2,)` |
| 给元组某个位置重新赋值 | 元组不可变，只能造新元组 |
| 左侧解包名字个数与元素不一致 | 名字个数必须与元素个数相同 |

最后补两条前面没有单独演示过的规则：`lambda` 没有 `return` 关键字；内层函数返回后仍然记得外层变量。
下面第二个例子用 `make_prod` 造出两个不同的乘法函数，互不干扰。

In [ ]:
double = lambda x: x * 2          # 没有 return 关键字
print(double(21))

def make_prod(a):
    def g(b):
        return a*b
    return g

tripler = make_prod(3)
print(tripler(4), make_prod(10)(4))